# Notebook 03 — AI Phishing Email Generation

Generates AI-generated phishing emails locally using Qwen2.5 3B via Ollama across five realistic phishing scenarios.

## What This Notebook Does
- Connects to locally running Ollama instance
- Defines five phishing generation scenarios:
  - Bank account suspension
  - IT helpdesk credential request
  - Prize notification
  - Invoice fraud
  - Delivery failure
- Generates 500 emails per scenario (2,492 total)
- Auto-saves progress every 50 emails to prevent data loss

## Prerequisites
- Ollama must be running locally
- Qwen2.5 3B must be downloaded: ollama pull qwen2.5:3b
- Start Ollama: ollama serve

## Outputs
- data/generated/bank_suspension.csv
- data/generated/it_helpdesk.csv
- data/generated/prize_notification.csv
- data/generated/invoice_fraud.csv
- data/generated/delivery_failure.csv
- data/generated/ai_phishing_all.csv (combined)

## Runtime
Approximately 14 hours
Output files are already saved in data/generated/

Generation uses temperature=0.9 for stylistic variety.


In [ ]:
import requests
import json
import pandas as pd
import time
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = Path("C:/phishing_detection")
DATA_GENERATED = BASE_DIR / "data" / "generated"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Verify Ollama is running
try:
    response = requests.get("http://localhost:11434/api/tags")
    models = [m['name'] for m in response.json()['models']]
    print("Ollama running. Available models:")
    for m in models:
        print(f"  - {m}")
except Exception as e:
    print(f"ERROR: Ollama not running — {e}")
    print("Start it with:")
    print('"C:\\Users\\hp\\AppData\\Local\\Programs\\Ollama\\ollama.exe" serve')

In [ ]:
# Five phishing scenarios - each produces a different style of attack
scenarios = [
    {
        "name": "bank_suspension",
        "count": 800,
        "prompt_template": """Write a convincing phishing email pretending to be from a bank.
The email should claim the recipient's account has been suspended due to suspicious activity.
Ask them to click a link to verify their identity and restore access.
Make it sound urgent and professional.
Write ONLY the email body, no subject line, no explanation.
Vary the bank name, the specific reason, and the tone each time."""
    },
    {
        "name": "it_helpdesk",
        "count": 800,
        "prompt_template": """Write a convincing phishing email pretending to be from an IT helpdesk or IT support team.
The email should claim the recipient's password is expiring or their account needs verification.
Ask them to click a link or reply with their credentials.
Make it sound like an internal company communication.
Write ONLY the email body, no subject line, no explanation.
Vary the company name, urgency level, and specific request each time."""
    },
    {
        "name": "prize_notification",
        "count": 800,
        "prompt_template": """Write a convincing phishing email claiming the recipient has won a prize, lottery, or reward.
The email should ask them to click a link or provide personal details to claim their prize.
Make it sound exciting but also credible.
Write ONLY the email body, no subject line, no explanation.
Vary the prize type, organization name, and instructions each time."""
    },
    {
        "name": "invoice_fraud",
        "count": 800,
        "prompt_template": """Write a convincing phishing email pretending to be an invoice or payment request from a legitimate business.
The email should pressure the recipient to pay an invoice or confirm payment details.
Make it sound like a routine business transaction.
Write ONLY the email body, no subject line, no explanation.
Vary the company name, invoice amount, and payment method each time."""
    },
    {
        "name": "delivery_failure",
        "count": 800,
        "prompt_template": """Write a convincing phishing email pretending to be from a delivery company like FedEx, UPS, or DHL.
The email should claim a package could not be delivered and ask the recipient to click a link to reschedule.
Make it sound like a standard delivery notification.
Write ONLY the email body, no subject line, no explanation.
Vary the delivery company, package details, and instructions each time."""
    }
]

total = sum(s['count'] for s in scenarios)
print(f"Scenarios defined: {len(scenarios)}")
print(f"Emails to generate: {total} total")
for s in scenarios:
    print(f"  - {s['name']}: {s['count']} emails")

In [ ]:
#single email generation function
def generate_phishing_email(prompt, model="qwen2.5:3b", max_retries=3):
    """Generate a single phishing email using the local LLM."""
    for attempt in range(max_retries):
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.9,  # High temperature = more variety
                        "top_p": 0.9,
                        "max_tokens": 300
                    }
                },
                timeout=30
            )
            if response.status_code == 200:
                text = response.json()["response"].strip()
                # Must be substantial enough to be a real email
                if len(text) > 100:
                    return text
        except Exception as e:
            if attempt == max_retries - 1:
                return None
            time.sleep(2)
    return None

# Test with one email from each scenario
print("Testing generation for each scenario\n")
for scenario in scenarios:
    result = generate_phishing_email(scenario['prompt_template'])
    if result:
        print(f" {scenario['name']} ({len(result)} chars)")
        print(f"  Preview: {result[:100]}...")
    else:
        print(f" {scenario['name']} — FAILED")
    print()

In [ ]:
import os

def generate_batch(scenario, save_path, model="qwen2.5:3b"):
    """Generate emails for one scenario with auto-save every 50 emails."""
    
    # Check if partial progress exists
    if save_path.exists():
        existing = pd.read_csv(save_path)
        already_done = len(existing)
        print(f"  Resuming from email {already_done}/{scenario['count']}")
        results = existing.to_dict('records')
    else:
        already_done = 0
        results = []
    
    remaining = scenario['count'] - already_done
    
    if remaining <= 0:
        print(f"  Already complete: {already_done} emails")
        return pd.DataFrame(results)
    
    print(f"  Generating {remaining} emails...")
    
    for i in tqdm(range(remaining), desc=scenario['name']):
        email_text = generate_phishing_email(scenario['prompt_template'], model)
        
        if email_text:
            # Clean up - remove subject lines if model added them
            lines = email_text.split('\n')
            cleaned_lines = [l for l in lines if not l.lower().startswith('subject:')]
            email_text = '\n'.join(cleaned_lines).strip()
            
            results.append({
                'text': email_text,
                'label': 2,  # AI-generated phishing
                'scenario': scenario['name'],
                'length': len(email_text)
            })
        
        # Auto-save every 50 emails
        if (i + 1) % 50 == 0:
            pd.DataFrame(results).to_csv(save_path, index=False)
    
    # Final save
    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False)
    return df

# Test with just 3 emails per scenario first to confirm everything works
print("Running small test (3 emails per scenario)")
test_results = []

for scenario in scenarios:
    test_scenario = scenario.copy()
    test_scenario['count'] = 3
    save_path = DATA_GENERATED / f"test_{scenario['name']}.csv"
    df = generate_batch(test_scenario, save_path)
    test_results.append(df)
    print(f"  ✓ {scenario['name']}: {len(df)} emails generated")

test_df = pd.concat(test_results, ignore_index=True)
print(f"\nTest complete: {len(test_df)} emails total")
print(test_df[['scenario', 'length']].groupby('scenario').mean().round(0))

In [ ]:
import requests
r = requests.get("http://localhost:11434/api/tags")
print(r.json())

In [ ]:
import os
from pathlib import Path

DATA_GENERATED = Path("C:/phishing_detection/data/generated")
test_files = list(DATA_GENERATED.glob("test_*.csv"))
for f in test_files:
    os.remove(f)
    print(f"Deleted: {f.name}")

In [ ]:
# Adjust count to 500 per scenario (2500 total)
for s in scenarios:
    s['count'] = 500

total = sum(s['count'] for s in scenarios)
print(f"Total emails to generate: {total}")
print("Progress auto-saves every 50 emails\n")

all_results = []

for scenario in scenarios:
    print(f"\n{'='*50}")
    print(f"Starting: {scenario['name']}")
    print(f"{'='*50}")
    
    save_path = DATA_GENERATED / f"{scenario['name']}.csv"
    df = generate_batch(scenario, save_path)
    all_results.append(df)
    
    print(f"Completed: {len(df)} emails saved to {save_path}")

# Combine all scenarios
final_df = pd.concat(all_results, ignore_index=True)

print(f"GENERATION COMPLETE")
print(f"Total emails generated: {len(final_df)}")
print(f"\nBreakdown by scenario:")
print(final_df['scenario'].value_counts())
print(f"\nAverage email length by scenario:")
print(final_df.groupby('scenario')['length'].mean().round(0))

# Save combined file
combined_path = DATA_GENERATED / "ai_phishing_all.csv"
final_df.to_csv(combined_path, index=False)
print(f"\nCombined file saved to: {combined_path}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("C:/phishing_detection")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_GENERATED = BASE_DIR / "data" / "generated"

# Load the finalised dataset
dataset_final = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")
print(f"Dataset loaded: {len(dataset_final)} emails")
print(dataset_final['label'].value_counts().sort_index())

In [ ]:
import nltk
import spacy
import textstat
import string
import re
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

nlp = spacy.load("en_core_web_sm")

# Copy all four feature functions from Notebook 02
def extract_basic_features(text):
    text = str(text)
    words = text.split()
    sentences = nltk.sent_tokenize(text)
    num_words = len(words) if len(words) > 0 else 1
    num_sentences = len(sentences) if len(sentences) > 0 else 1
    return {
        'email_length': len(text),
        'num_words': len(words),
        'num_sentences': num_sentences,
        'num_paragraphs': len([p for p in text.split('\n\n') if p.strip()]),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'max_word_length': max([len(w) for w in words]) if words else 0,
        'avg_sentence_length': num_words / num_sentences,
        'unique_words': len(set(words)),
        'type_token_ratio': len(set(words)) / num_words,
        'vocab_richness': len(set(w.lower() for w in words)) / num_words,
        'num_exclamations': text.count('!'),
        'num_questions': text.count('?'),
        'num_commas': text.count(','),
        'num_periods': text.count('.'),
        'exclamation_ratio': text.count('!') / num_words,
        'question_ratio': text.count('?') / num_words,
        'num_capitals': sum(1 for c in text if c.isupper()),
        'capital_ratio': sum(1 for c in text if c.isupper()) / len(text) if text else 0,
        'num_special_chars': sum(1 for c in text if c in string.punctuation),
        'special_char_ratio': sum(1 for c in text if c in string.punctuation) / len(text) if text else 0,
    }

def extract_phishing_features(text):
    text = str(text)
    words = text.split()
    num_words = len(words) if len(words) > 0 else 1
    text_lower = text.lower()
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    urls = url_pattern.findall(text)
    urgency_words = ['urgent', 'immediately', 'expire', 'suspend', 'verify',
                     'confirm', 'update', 'click', 'login', 'password',
                     'account', 'bank', 'limited', 'offer', 'winner', 'prize',
                     'free', 'congratulations', 'selected', 'act now']
    greeting_words = ['dear', 'hello', 'hi', 'greetings', 'good morning',
                      'good afternoon', 'dear customer', 'dear user']
    threat_words = ['suspended', 'terminated', 'blocked', 'restricted',
                    'unauthorized', 'illegal', 'fraud', 'risk']
    return {
        'num_urls': len(urls),
        'has_url': int(len(urls) > 0),
        'url_ratio': len(urls) / num_words,
        'num_http': text_lower.count('http://'),
        'num_https': text_lower.count('https://'),
        'urgency_word_count': sum(1 for w in urgency_words if w in text_lower),
        'threat_word_count': sum(1 for w in threat_words if w in text_lower),
        'has_greeting': int(any(g in text_lower for g in greeting_words)),
        'has_unsubscribe': int('unsubscribe' in text_lower),
        'has_dear': int('dear' in text_lower),
        'has_winner': int('winner' in text_lower or 'won' in text_lower),
        'has_free': int('free' in text_lower),
        'has_click_here': int('click here' in text_lower),
        'has_verify': int('verify' in text_lower or 'verification' in text_lower),
        'has_account': int('account' in text_lower),
        'has_password': int('password' in text_lower),
        'has_bank': int('bank' in text_lower),
        'has_invoice': int('invoice' in text_lower or 'payment' in text_lower),
        'num_digits': sum(c.isdigit() for c in text),
        'digit_ratio': sum(c.isdigit() for c in text) / len(text) if text else 0,
    }

def extract_readability_features(text):
    text = str(text)
    if len(text.split()) < 10:
        return {k: 0 for k in ['flesch_reading_ease', 'flesch_kincaid_grade',
                                'gunning_fog', 'smog_index', 'coleman_liau_index',
                                'automated_readability_index', 'dale_chall_readability',
                                'difficult_words', 'linsear_write_formula', 'text_standard']}
    try:
        return {
            'flesch_reading_ease': textstat.flesch_reading_ease(text),
            'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
            'gunning_fog': textstat.gunning_fog(text),
            'smog_index': textstat.smog_index(text),
            'coleman_liau_index': textstat.coleman_liau_index(text),
            'automated_readability_index': textstat.automated_readability_index(text),
            'dale_chall_readability': textstat.dale_chall_readability_score(text),
            'difficult_words': textstat.difficult_words(text),
            'linsear_write_formula': textstat.linsear_write_formula(text),
            'text_standard': float(str(textstat.text_standard(text, float_output=True))),
        }
    except:
        return {k: 0 for k in ['flesch_reading_ease', 'flesch_kincaid_grade',
                                'gunning_fog', 'smog_index', 'coleman_liau_index',
                                'automated_readability_index', 'dale_chall_readability',
                                'difficult_words', 'linsear_write_formula', 'text_standard']}

def extract_syntactic_features(text):
    text = str(text)
    if len(text) > 5000:
        text = text[:5000]
    doc = nlp(text)
    total_tokens = len(doc) if len(doc) > 0 else 1
    pos_counts = {}
    for token in doc:
        pos_counts[token.pos_] = pos_counts.get(token.pos_, 0) + 1
    return {
        'noun_ratio': pos_counts.get('NOUN', 0) / total_tokens,
        'verb_ratio': pos_counts.get('VERB', 0) / total_tokens,
        'adj_ratio': pos_counts.get('ADJ', 0) / total_tokens,
        'adv_ratio': pos_counts.get('ADV', 0) / total_tokens,
        'pronoun_ratio': pos_counts.get('PRON', 0) / total_tokens,
        'propn_ratio': pos_counts.get('PROPN', 0) / total_tokens,
        'det_ratio': pos_counts.get('DET', 0) / total_tokens,
        'punct_ratio': pos_counts.get('PUNCT', 0) / total_tokens,
        'num_ratio': pos_counts.get('NUM', 0) / total_tokens,
        'num_entities': len(doc.ents),
        'entity_ratio': len(doc.ents) / total_tokens,
        'stopword_ratio': sum(1 for token in doc if token.is_stop) / total_tokens,
        'unique_punct': len(set(token.text for token in doc if token.is_punct)),
    }

def extract_all_features(text):
    features = {}
    features.update(extract_basic_features(text))
    features.update(extract_phishing_features(text))
    features.update(extract_readability_features(text))
    features.update(extract_syntactic_features(text))
    return features

print("All feature functions loaded.")
print(f"Total emails to process: {len(dataset_final)}")

In [ ]:
print("Extracting stylometric features for all emails")

all_features = []
for idx, row in tqdm(dataset_final.iterrows(), total=len(dataset_final), desc="Extracting"):
    try:
        features = extract_all_features(row['text'])
        features['label'] = row['label']
        all_features.append(features)
    except Exception as e:
        print(f"Skipped email {idx}: {e}")
        continue

features_df = pd.DataFrame(all_features)

print(f"Shape: {features_df.shape}")
print(f"Emails processed: {len(features_df)}")
print(f"Features per email: {features_df.shape[1] - 1}")
print(f"Missing values: {features_df.isnull().sum().sum()}")

# Save
save_path = DATA_PROCESSED / "stylometric_features_final.csv"
features_df.to_csv(save_path, index=False)
print(f"\nSaved to: {save_path}")
print(f"File size: {save_path.stat().st_size / (1024*1024):.1f} MB")

In [ ]:
from pathlib import Path
DATA_PROCESSED = Path("C:/phishing_detection/data/processed")
save_path = DATA_PROCESSED / "stylometric_features_final.csv"

if save_path.exists():
    import pandas as pd
    df = pd.read_csv(save_path)
    print(f"Progress: {len(df)} emails processed so far")
else:
    print("File not created yet - still processing first batch")